# Decision Models as Tools

An Agents SDK agent can delegate a question to a function tool powered by a decision model. Luna writes the question; the tool passes it directly to Microsoft Decision-1. Decision-1 selects backend operations, Python executes them, and Luna interprets the returned evidence.

We will compare this agent with an LLM-only SDK agent using the same backend functions. Each model selects at most one tool action per turn. We will inspect the answers, elapsed time, token usage, and estimated Azure cost.


## 1. Set up Azure and the Agents SDK

Install requirements.txt, select this project's virtual environment as the notebook kernel, and put the Azure endpoint, key, and deployment names in .env. Runs make paid API requests.

Both agents use GPT-6-Luna with the same settings. Agents SDK manages their model calls, tool execution, and conversation history. Tracing export is disabled for this local demo.

Both agents use medium reasoning effort. The 8,192-token output allowance covers reasoning and the visible response; actual usage determines cost.

Two HTTP hooks record Luna request timings, response status, and retry count. End-to-end timing still covers the complete SDK run.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
from statistics import median
from time import perf_counter
import asyncio
import json
import os
import ssl

import httpx
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display
from openai import AsyncOpenAI, DefaultAsyncHttpxClient
from openai.types.shared import Reasoning
from agents import (
    Agent, ModelSettings, OpenAIResponsesModel, RunContextWrapper,
    Runner, function_tool, set_tracing_disabled,
)

load_dotenv(Path.cwd() / ".env", override=True)
AZURE_ENDPOINT = os.environ["AZURE_ENDPOINT"].rstrip("/")
DECISION_DEPLOYMENT = os.environ["AZURE_DECISION_DEPLOYMENT"]
LLM_DEPLOYMENT = os.environ["AZURE_TEXT_DEPLOYMENT"]

set_tracing_disabled(True)
llm_http_calls = []


async def time_llm_request(request):
    request.extensions["started_s"] = perf_counter()


async def time_llm_response(response):
    await response.aread()
    started = response.request.extensions["started_s"]
    seconds = perf_counter() - started
    llm_http_calls.append({
        "started_s": started, "elapsed_s": seconds,
        "status": response.status_code,
        "retry_count": int(response.request.headers["x-stainless-retry-count"]),
    })
    print(f"Luna HTTP: {seconds:.2f} s", flush=True)


azure = AsyncOpenAI(
    api_key=os.environ["AZURE_API_KEY"],
    base_url=AZURE_ENDPOINT + "/openai/v1/",
    timeout=60, max_retries=2,
    http_client=DefaultAsyncHttpxClient(
        verify=ssl.create_default_context(),
        event_hooks={"request": [time_llm_request], "response": [time_llm_response]},
    ),
)
model = OpenAIResponsesModel(model=LLM_DEPLOYMENT, openai_client=azure)
settings = ModelSettings(
    parallel_tool_calls=False,
    reasoning=Reasoning(effort="medium"),
    max_tokens=8192,
    store=False,
    preserve_raw_usage=True,
)


## 2. Define the request and fictional backend

The customer needs an order by Friday. The records below contain warehouse notes, carrier events, inventory, shipping estimates, and policy. The models must connect those facts; the fixture contains no prepared diagnosis or recommended answer.

The backend functions are local and add no artificial delays. Both agents use these exact functions.


In [2]:
AS_OF = "2026-10-21T10:00:00+02:00"
CUSTOMER_REQUEST = (
    "My order ORD-1042 still hasn't shipped, and I need it by Friday, October 23. "
    "Could you find out what's happening and whether there's a way to get it here in time?"
)

ORDERS = {"ORD-1042": {
    "order_id": "ORD-1042", "item": "Arc desk lamp", "sku": "LAMP-01", "quantity": 1,
    "destination": "CH-8000", "warehouse_id": "NORTH", "tracking_id": "TRK-1042",
    "status": "processing", "promised_delivery_date": "2026-10-23",
}}
TRACKING = {"TRK-1042": {
    "tracking_id": "TRK-1042",
    "events": [{"timestamp": "2026-10-20T15:40:00+02:00", "event": "Shipping label created"}],
    "carrier_acceptance_scan": None, "estimated_delivery_date": None,
}}
FULFILLMENT_NOTES = {"ORD-1042": {
    "order_id": "ORD-1042", "warehouse_id": "NORTH",
    "notes": [
        "October 20, 16:10: Picking paused. The allocated batch failed its quality check; "
        "units were quarantined.",
        "October 21, 08:20: No replacement units here. Replenishment expected Saturday, "
        "October 24. Parcel has not been handed to the carrier.",
    ],
}}
INVENTORY = {"LAMP-01": {
    "sku": "LAMP-01", "checked_at": AS_OF,
    "warehouses": [
        {"warehouse_id": "NORTH", "available_quantity": 0, "next_stock_date": "2026-10-24"},
        {"warehouse_id": "WEST", "available_quantity": 1, "next_stock_date": None},
        {"warehouse_id": "REMOTE", "available_quantity": 5, "next_stock_date": None},
    ],
}}
SHIPPING = {
    "NORTH": [
        {"service": "standard", "estimated_delivery_date": "2026-10-28", "incremental_cost_eur": 0},
        {"service": "express", "estimated_delivery_date": "2026-10-26", "incremental_cost_eur": 12},
    ],
    "WEST": [
        {"service": "standard", "estimated_delivery_date": "2026-10-26", "incremental_cost_eur": 0},
        {"service": "express", "estimated_delivery_date": "2026-10-23", "incremental_cost_eur": 12},
    ],
    "REMOTE": [
        {"service": "standard", "estimated_delivery_date": "2026-10-28", "incremental_cost_eur": 0},
        {"service": "express", "estimated_delivery_date": "2026-10-26", "incremental_cost_eur": 18},
    ],
}
POLICY = {"policy_id": "fulfillment-2026-10", "rules": [
    "Before carrier handover, an order may be rerouted to another warehouse for the same SKU.",
    "If a warehouse delay threatens the promised delivery date, an express surcharge up to "
    "EUR 15 is eligible for a waiver.",
    "An option requires available stock and a valid shipping quote. Delivery dates are estimates.",
    "A lookup does not reserve stock, approve a waiver, or change the order.",
]}


In [3]:
def get_order_details(order_id: str) -> dict:
    """Read order details, status, and identifiers."""
    print("  get_order_details", order_id, flush=True)
    return ORDERS[order_id]


def get_tracking_events(tracking_id: str) -> dict:
    """Read carrier events and any delivery estimate."""
    print("  get_tracking_events", tracking_id, flush=True)
    return TRACKING[tracking_id]


def get_fulfillment_notes(order_id: str) -> dict:
    """Read warehouse fulfillment notes."""
    print("  get_fulfillment_notes", order_id, flush=True)
    return FULFILLMENT_NOTES[order_id]


def get_alternative_inventory(sku: str) -> dict:
    """Read available same-SKU stock by warehouse."""
    print("  get_alternative_inventory", sku, flush=True)
    return INVENTORY[sku]


def get_shipping_options(warehouse_id: str, destination: str) -> dict:
    """Read shipping estimates, incremental charges, and dispatch cutoff."""
    print("  get_shipping_options", warehouse_id, destination, flush=True)
    return {
        "warehouse_id": warehouse_id, "destination": destination, "quoted_at": AS_OF,
        "valid_until": "2026-10-21T14:00:00+02:00",
        "dispatch_cutoff": "2026-10-21T14:00:00+02:00",
        "quotes": SHIPPING[warehouse_id],
        "note": "NORTH quotes assume October 24 replenishment; other locations can dispatch today.",
    }


def get_fulfillment_policy() -> dict:
    """Read rerouting and shipping surcharge policy."""
    print("  get_fulfillment_policy", flush=True)
    return POLICY


## 3. Prepare operations the decision model can select

Each choice pairs a backend function with concrete arguments. Python prepares these choices from the order ID and observed records. It does not need to interpret the delegated question.

Once inventory is known, shipping checks are offered for every listed warehouse, including those with no stock. Completed lookups are omitted. Decision-1 uses the question and evidence to choose which operation is useful.


In [4]:
def observed(evidence, name, arguments):
    return next((entry["result"] for entry in evidence
                 if entry["name"] == name and entry["arguments"] == arguments), None)


def available_actions(order_id, evidence):
    candidates = [
        (get_order_details, {"order_id": order_id}),
        (get_fulfillment_notes, {"order_id": order_id}),
        (get_fulfillment_policy, {}),
    ]
    order = observed(evidence, "get_order_details", {"order_id": order_id})
    if order:
        candidates += [
            (get_tracking_events, {"tracking_id": order["tracking_id"]}),
            (get_alternative_inventory, {"sku": order["sku"]}),
        ]
        inventory = observed(evidence, "get_alternative_inventory", {"sku": order["sku"]})
        if inventory:
            candidates += [
                (get_shipping_options, {
                    "warehouse_id": warehouse["warehouse_id"], "destination": order["destination"]})
                for warehouse in inventory["warehouses"]
            ]
    return {
        f"a{index}": (function, arguments)
        for index, (function, arguments) in enumerate(candidates, 1)
        if observed(evidence, function.__name__, arguments) is None
    }


## 4. Ask Decision-1 to select one operation

Decision-1 receives the delegated question, current evidence, and available choices. It returns an action ID or a stopping choice. The instructions contain no fixture-specific answer or preferred action.

This is the only custom model request helper. It retries Azure transport errors and HTTP 408/429/500/502/503/504 up to twice. GPT-6-Luna requests use the OpenAI client's built-in retries.


In [5]:
async def choose_decision(http, question, evidence, actions):
    choices = {
        key: f"{function.__name__}({json.dumps(arguments)}): {function.__doc__}"
        for key, (function, arguments) in actions.items()
    }
    choices["done"] = "Return the observed evidence: it answers the delegated question."
    choices["blocked"] = "Return the observed evidence: available operations cannot answer the question."
    body = {
        "model": DECISION_DEPLOYMENT,
        "state": {"question": question, "as_of": AS_OF, "evidence": evidence},
        "questions": {"action": {
            "type": "choice",
            "instructions": (
                "Choose the next useful operation for this delegated question: " + question +
                " Use the observed records and collect missing evidence. "
                "Choose done when the evidence answers the question, or blocked if no "
                "available operation can obtain a needed fact."
            ),
            "criteria": choices,
        }},
    }
    started = perf_counter()
    for attempt in range(3):
        try:
            response = await http.post(
                AZURE_ENDPOINT + "/providers/microsoft/v1/systemone",
                json=body, headers={"api-key": os.environ["AZURE_API_KEY"]})
            response.raise_for_status()
            result = response.json()
            break
        except (httpx.HTTPStatusError, httpx.TransportError) as error:
            if attempt == 2 or (isinstance(error, httpx.HTTPStatusError)
                    and error.response.status_code not in {408, 429, 500, 502, 503, 504}):
                raise
            print(f"Azure decision request retry {attempt + 1}/2", flush=True)
            await asyncio.sleep(2 ** attempt)
    return result["answers"]["action"]["choice"], {
        "latency_ms": (perf_counter() - started) * 1000,
        "http_attempts": attempt + 1, "usage": result["usage"],
    }


## 5. Package the decision loop as an SDK function tool

The outer LLM supplies an order ID and writes the question it wants answered. The question is passed unchanged to Decision-1. Inside the tool, Python prepares executable choices from known identifiers, asks Decision-1 to select one, executes it, and adds the returned records to the evidence.

RunContextWrapper supplies a plain per-run dictionary. It retains evidence between tool invocations and collects decision usage for the cost calculation.


In [6]:
@function_tool
async def order_specialist(
    context: RunContextWrapper[dict],
    order_id: str,
    question: str,
) -> dict:
    """Gather source records to answer an order-support question using an internal decision model.

    Args:
        order_id: The customer's order identifier.
        question: The question to answer from order-support records, including relevant customer constraints.
    """
    evidence = context.context["evidence"]
    print(f"Specialist question: {question}", flush=True)
    async with httpx.AsyncClient(verify=ssl.create_default_context(), timeout=60) as http:
        while True:
            actions = available_actions(order_id, evidence)
            choice, stats = await choose_decision(http, question, evidence, actions)
            name = choice if choice in {"done", "blocked"} else actions[choice][0].__name__
            context.context["decisions"].append({"question": question, "action": name, **stats})
            if choice in {"done", "blocked"}:
                print("  Decision-1:", choice, flush=True)
                return {"question": question, "status": choice, "evidence": list(evidence)}
            function, arguments = actions[choice]
            result = function(**arguments)
            evidence.append({"name": function.__name__, "arguments": arguments, "result": result})


## 6. Configure the two SDK agents

Both agents receive exactly the same instructions, model, and model settings. The hybrid sees only order_specialist and chooses what question to delegate. The LLM-only agent sees the six underlying functions, wrapped with the SDK's function_tool helper.

parallel_tool_calls=False allows at most one tool call per LLM response. Inside the specialist, each decision request selects one operation. Agents SDK owns both outer agent loops.


In [7]:
INSTRUCTIONS = (
    "You are an order-support assistant. Help the customer using the available tools. "
    "Base your answer on returned records. Treat shipping dates as estimates, and "
    "describe actions that have not been performed as proposals. "
    f"The current fictional time is {AS_OF}."
)

hybrid_agent = Agent(
    name="Order support with a decision-model tool",
    instructions=INSTRUCTIONS, model=model, model_settings=settings,
    tools=[order_specialist],
)
llm_agent = Agent(
    name="Order support with backend tools",
    instructions=INSTRUCTIONS, model=model, model_settings=settings,
    tools=[
        function_tool(get_order_details),
        function_tool(get_tracking_events),
        function_tool(get_fulfillment_notes),
        function_tool(get_alternative_inventory),
        function_tool(get_shipping_options),
        function_tool(get_fulfillment_policy),
    ],
)


## 7. Run the LLM-only agent

This agent can use the same backend operations and data. It chooses its own sequence, with one tool call per LLM response. It receives no instruction to imitate the specialist's path.


In [8]:
llm_http_calls.clear()
llm_started_at = datetime.now(timezone.utc).isoformat()
started = perf_counter()
llm_result = await Runner.run(llm_agent, CUSTOMER_REQUEST)
llm_seconds = perf_counter() - started
llm_only_http_calls = list(llm_http_calls)
display(Markdown(llm_result.final_output))


Luna HTTP: 2.95 s


  get_order_details ORD-1042


Luna HTTP: 1.39 s


  get_fulfillment_notes ORD-1042


Luna HTTP: 5.76 s


...

  get_shipping_options WEST CH-8000


Luna HTTP: 1.49 s


  get_fulfillment_policy


Luna HTTP: 7.89 s


Your order hasn’t shipped because the allocated batch failed its quality check and was quarantined. The NORTH warehouse has no replacement units; replenishment is expected October 24, so the current route won’t meet your October 23 date.

There may be a way to meet it: WEST has one unit available, and its **express** quote estimates delivery to CH-8000 on **Friday, October 23**, with a **€12 surcharge**. That date is an estimate, not a guarantee. The surcharge is within the policy limit for a possible waiver, but a waiver hasn’t been approved.

I can propose rerouting the order to WEST and requesting the surcharge waiver. Neither the reroute nor the waiver has been applied, and the stock isn’t reserved. The quote and dispatch cutoff are **today at 14:00**, so please let me know promptly if you’d like me to proceed.

## 8. Run the hybrid agent

The call to Runner.run below is the entire outer agent execution. Read the backend operations inside each specialist invocation, then inspect the final answer against the source records.


In [9]:
llm_http_calls.clear()
hybrid_context = {"evidence": [], "decisions": []}
hybrid_started_at = datetime.now(timezone.utc).isoformat()
started = perf_counter()
hybrid_result = await Runner.run(
    hybrid_agent, CUSTOMER_REQUEST, context=hybrid_context,
)
hybrid_seconds = perf_counter() - started
hybrid_http_calls = list(llm_http_calls)
display(Markdown(hybrid_result.final_output))


Luna HTTP: 1.71 s


Specialist question: Customer says order has not shipped and needs it by Friday, October 23, 2026. Find current fulfillment status, reason for delay, estimated delivery/timing, and available options/actions to improve chance of arrival by then. Clarify any dates as estimates and actions not performed as proposals.


  get_order_details ORD-1042


  get_fulfillment_notes ORD-1042


  get_alternative_inventory LAMP-01


  get_shipping_options WEST CH-8000


  get_shipping_options REMOTE CH-8000


  Decision-1: done


Luna HTTP: 4.03 s


Your order is still **processing** and hasn’t been handed to the carrier. The allocated lamp batch failed its quality check and was quarantined; the North warehouse has no replacement stock until **Saturday, October 24**. So the original **October 23** delivery estimate is at risk.

There is another possibility: the **West warehouse has one available**. Its **express** option to CH-8000 is estimated to arrive **Friday, October 23**, with an additional cost of **€12**. That’s an estimate, not a guarantee. The quote is valid until **14:00 today**, and the dispatch cutoff is also **14:00 today**.

I haven’t changed the order. Would you like me to request switching fulfillment to West and using express shipping for the additional €12?

## 9. Calculate Azure model cost

The confirmed deployment type is Global Standard. These are published USD rates per million tokens, checked on 16 October 2026:

| Model | Input | Cached input | Cache writes | Output |
|---|---:|---:|---:|---:|
| GPT-6-Luna, short context | 0.10 | 0.01 | 0.125 | 0.50 |
| Microsoft Decision-1 | 0.042 | -- | -- | 0 |

The calculation uses SDK-reported input, cached input, cache-write, and output tokens for the LLM, plus native usage from every decision request, including stopping choices. Cached reads and cache writes are subsets of the reported LLM input total; ordinary input is the remainder. Decision input is charged at the published input rate.

These are usage-based estimates of model cost. Both the outer LLM and internal decision calls count toward the hybrid total. The local backend adds no service charges.

Sources: [Azure GPT-6 rates](https://azure.microsoft.com/en-us/blog/gpt-6-astra-sol-and-luna-for-production-agents-in-microsoft-foundry/), [Microsoft Decision-1 rates](https://commandline.microsoft.com/microsoft-decision-1-model-foundry/).

Reported output tokens include reasoning tokens, so they are already included in the output charge. The summary also displays reasoning tokens separately.


In [10]:
LUNA_RATES = {"input": 0.10, "cached": 0.01, "cache_write": 0.125, "output": 0.50}
DECISION_INPUT_RATE = 0.042


def summarize(result, seconds, decisions, backend_operations, http_calls):
    usage = result.context_wrapper.usage
    cached = usage.input_tokens_details.cached_tokens
    writes = usage.input_tokens_details.cache_write_tokens
    ordinary = usage.input_tokens - cached - writes
    llm_cost = (
        ordinary * LUNA_RATES["input"] + cached * LUNA_RATES["cached"]
        + writes * LUNA_RATES["cache_write"] + usage.output_tokens * LUNA_RATES["output"]
    ) / 1_000_000
    decision_input = sum(call["usage"]["input_tokens"] for call in decisions)
    decision_cost = decision_input * DECISION_INPUT_RATE / 1_000_000
    llm_http_s = sum(call["elapsed_s"] for call in http_calls)
    decision_http_s = sum(call["latency_ms"] for call in decisions) / 1000
    return {
        "llm_requests": len(result.raw_responses),
        "decision_requests": len(decisions),
        "backend_operations": backend_operations,
        "elapsed_s": seconds,
        "llm_http_s": llm_http_s,
        "decision_http_s": decision_http_s,
        "sdk_and_other_s": seconds - llm_http_s - decision_http_s,
        "llm_input_tokens": usage.input_tokens,
        "llm_cached_tokens": cached,
        "llm_cache_write_tokens": writes,
        "llm_output_tokens": usage.output_tokens,
        "llm_reasoning_tokens": usage.output_tokens_details.reasoning_tokens,
        "decision_input_tokens": decision_input,
        "decision_median_ms": median(call["latency_ms"] for call in decisions) if decisions else None,
        "llm_cost_usd": llm_cost,
        "decision_cost_usd": decision_cost,
        "total_cost_usd": llm_cost + decision_cost,
    }


hybrid_summary = summarize(
    hybrid_result, hybrid_seconds, hybrid_context["decisions"],
    len(hybrid_context["evidence"]), hybrid_http_calls,
)
llm_calls = [item for item in llm_result.new_items if item.type == "tool_call_item"]
llm_summary = summarize(llm_result, llm_seconds, [], len(llm_calls), llm_only_http_calls)

lines = [
    "| Architecture | LLM requests | Decision requests | Backend calls | Elapsed | Model cost (USD) |",
    "|---|---:|---:|---:|---:|---:|",
]
for label, summary in [("Decision model as a tool", hybrid_summary), ("Sequential LLM agent", llm_summary)]:
    lines.append(
        f"| {label} | {summary['llm_requests']} | {summary['decision_requests']} | "
        f"{summary['backend_operations']} | {summary['elapsed_s']:.2f} s | "
        f"{summary['total_cost_usd']:.6f} |")
display(Markdown("\n".join(lines)))
print(json.dumps({"hybrid": hybrid_summary, "llm_only": llm_summary}, indent=2))

timing_lines = [
    "| Architecture | Luna HTTP | Decision requests | SDK and other time |",
    "|---|---:|---:|---:|",
]
for label, item in [("Decision model as a tool", hybrid_summary), ("Sequential LLM agent", llm_summary)]:
    timing_lines.append(
        f"| {label} | {item['llm_http_s']:.2f} s | {item['decision_http_s']:.2f} s | "
        f"{item['sdk_and_other_s']:.2f} s |")
display(Markdown("\n".join(timing_lines)))


| Architecture | LLM requests | Decision requests | Backend calls | Elapsed | Model cost (USD) |
|---|---:|---:|---:|---:|---:|
| Decision model as a tool | 2 | 6 | 5 | 8.88 s | 0.000529 |
| Sequential LLM agent | 6 | 0 | 5 | 25.44 s | 0.000703 |

{
  "hybrid": {
    "llm_requests": 2,
    "decision_requests": 6,
    "backend_operations": 5,
    "elapsed_s": 8.876828600012232,
    "llm_http_s": 5.7378807999775745,
    "decision_http_s": 2.9212839000392705,
    "sdk_and_other_s": 0.2176638999953866,
    "llm_input_tokens": 1453,
    "llm_cached_tokens": 0,
    "llm_cache_write_tokens": 1177,
    "llm_output_tokens": 376,
    "llm_reasoning_tokens": 121,
    "decision_input_tokens": 3962,
    "decision_median_ms": 310.02355000237003,
    "llm_cost_usd": 0.00036272500000000005,
    "decision_cost_usd": 0.000166404,
    "total_cost_usd": 0.000529129
  },
  "llm_only": {
    "llm_requests": 6,
    "decision_requests": 0,
    "backend_operations": 5,
    "elapsed_s": 25.439995700027794,
    "llm_http_s": 25.035294099943712,
    "decision_http_s": 0.0,
    "sdk_and_other_s": 0.4047016000840813,
    "llm_input_tokens": 4467,
    "llm_cached_tokens": 0,
    "llm_cache_write_tokens": 1114,
    "llm_output_tokens": 456,
    "llm_reasoning_

| Architecture | Luna HTTP | Decision requests | SDK and other time |
|---|---:|---:|---:|
| Decision model as a tool | 5.74 s | 2.92 s | 0.22 s |
| Sequential LLM agent | 25.04 s | 0.00 s | 0.40 s |

## 10. Inspect the SDK calls, evidence, and usage

The records below retain the actual questions Luna delegated, selected backend operations, source evidence, per-request timings, usage, and final answers. They omit hidden reasoning.

Elapsed time covers Runner.run, including model calls, retries, and tool execution, and excludes setup and display. Luna HTTP timings cover requests that received a response; the remaining time includes SDK work, local execution, and any other waiting. Native decision-request timings include retries and waits. Backend functions are local. Provider caching can affect timing and is reflected in the cost calculation.

Inspect both answers against their source records before comparing completion times. A selected illustrative pair demonstrates what happened in that example; it does not estimate typical performance.

The saved outputs are pair 2, selected for the largest observed speedup among three complete paired trials on 16 October 2026. Execution order alternated across the batch; LLM-only ran first in this pair. Both answers passed manual inspection in all three pairs, and the hybrid was faster in each. All pair timings are retained in README.md.

Manual inspection: both answers explain the quarantined batch and unavailable local stock, identify West express delivery estimated for October 23 at EUR 12, preserve delivery uncertainty, and state that no change was performed. Both used five backend lookups. Hybrid retrieved order details, warehouse notes, inventory, and West and Remote shipping quotes. LLM-only retrieved order details, warehouse notes, inventory, West shipping, and policy. Its answer discusses possible waiver eligibility; the hybrid makes no waiver claim.

This selected example took 8.88 s versus 25.44 s (2.87x speedup), with 24.7% lower estimated model cost. These figures describe the selected example. Rerunning the notebook produces a fresh pair in the displayed LLM-only-then-hybrid order.


In [11]:
def sdk_calls(result):
    return [
        {"name": item.raw_item.name, "arguments": json.loads(item.raw_item.arguments)}
        for item in result.new_items if item.type == "tool_call_item"
    ]


display(JSON({
    "configuration": {
        "sdk": "openai-agents 0.24.0", "llm_deployment": LLM_DEPLOYMENT,
        "specialist_arguments": ["order_id", "question"],
        "decision_deployment": DECISION_DEPLOYMENT,
        "parallel_tool_calls": False, "reasoning_effort": "medium", "max_tokens": 8192,
        "pricing": "Azure Global Standard, short context, USD; checked 2026-10-16",
        "luna_rates_per_million": LUNA_RATES, "decision_input_rate_per_million": DECISION_INPUT_RATE,
        "as_of": AS_OF, "request": CUSTOMER_REQUEST,
    },
    "hybrid": {
        "run_at_utc": hybrid_started_at, "summary": hybrid_summary,
        "sdk_calls": sdk_calls(hybrid_result), "evidence": hybrid_context["evidence"],
        "decisions": hybrid_context["decisions"],
        "llm_usage": [response.raw_usage for response in hybrid_result.raw_responses],
        "llm_http_calls": hybrid_http_calls,
        "answer": hybrid_result.final_output,
    },
    "llm_only": {
        "run_at_utc": llm_started_at, "summary": llm_summary,
        "sdk_calls": sdk_calls(llm_result),
        "tool_outputs": [item.output for item in llm_result.new_items
                         if item.type == "tool_call_output_item"],
        "llm_usage": [response.raw_usage for response in llm_result.raw_responses],
        "llm_http_calls": llm_only_http_calls,
        "answer": llm_result.final_output,
    },
}, expanded=False))


<IPython.core.display.JSON object>

## The reusable pattern

Luna writes a question for an SDK function tool. Inside that tool, Decision-1 chooses among executable backend operations prepared by Python. The tool returns source evidence for Luna to interpret. The same question-and-evidence interface can support different customer requests within the backend's capabilities.

[Agents SDK](https://developers.openai.com/api/docs/guides/agents/quickstart) | [Decision-1 API](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/how-to/use-foundry-models-microsoft-decision)
